In [ ]:
import sys
from pathlib import Path

for candidate in [Path("../Bronze"), Path(".")]:
    if (candidate / "bronze_json.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

from bronze_json import load_bronze

from pyspark.sql.functions import col, explode, explode_outer, from_json, lit, schema_of_json
import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, DoubleType, LongType, StringType, StructField, StructType, TimestampType


def parse_entity(bronze_df, entity_type):
    """Parse bronze JSON payloads for one entity_type into typed columns."""
    rows = bronze_df.filter(col("entity_type") == entity_type)
    sample = rows.select("payload").first()
    if sample is None:
        raise ValueError(f"No bronze rows for entity_type={entity_type}")

    schema = schema_of_json(lit(sample.payload))
    return (
        rows.select(from_json(col("payload"), schema).alias("record"))
        .select("record.*")
    )



In [3]:
bronze_all = load_bronze(spark)

# 🥈 Silver Layer — Data Cleaning & Transformation

## Overview
Silver reads the **single Bronze JSON table**, parses each `payload` JSON object by `entity_type`,
then builds clean typed tables in Unity Catalog.

Bronze path: `/Volumes/ecom_clickstream/bronze/delta/clickstream_json/`

##Users

The `users` table is the simplest transformation in the Silver Layer.
It contains 3 columns read from the Bronze Layer.

### Transformations Applied

**1. Type Casting — `user_first_touch_timestamp`**
The timestamp is stored as a `String` in Bronze, representing timestamp in **microseconds**.
We first cast it to `LongType`, then divide by 1,000,000 to convert microseconds to seconds,
and finally cast to `TimestampType`.


### Reading and displaying

In [4]:
df_users_bronze = parse_entity(bronze_all, "users").drop("_ingested_at", "_source_file")

df_users_bronze.printSchema()
display(df_users_bronze.limit(5))

root
 |-- user_id: string (nullable = true)
 |-- user_first_touch_timestamp: string (nullable = true)
 |-- email: string (nullable = true)



,user_id,user_first_touch_timestamp,email
0,UA000000102357305,1592182691348767,None
1,UA000000102357308,1592183287634953,None
2,UA000000102357309,1592183302736627,None
3,UA000000102357321,1592184604178702,david23@orozco-parker.com
4,UA000000102357325,1592185154063628,None


### Cast first touch column

In [5]:
df_users_silver =df_users_bronze\
    .withColumn("user_first_touch_timestamp", col("user_first_touch_timestamp").cast(LongType()))\
    .withColumn("user_first_touch_timestamp", (col("user_first_touch_timestamp") / 1000000).cast(TimestampType()))

df_users_silver.select('user_first_touch_timestamp')

,user_first_touch_timestamp
0,2020-06-15 00:58:11.348767
1,2020-06-15 01:08:07.634953
2,2020-06-15 01:08:22.736627
3,2020-06-15 01:30:04.178702
4,2020-06-15 01:39:14.063628
5,2020-06-15 01:55:22.660210
6,2020-06-15 01:58:20.091435
7,2020-06-15 02:21:03.145345
8,2020-06-15 02:22:12.257656
9,2020-06-15 02:31:51.375015


In [6]:
df_users_silver.select('user_first_touch_timestamp').show(5,truncate=False)

+--------------------------+
|user_first_touch_timestamp|
+--------------------------+
|2020-06-15 00:58:11.348767|
|2020-06-15 01:08:07.634953|
|2020-06-15 01:08:22.736627|
|2020-06-15 01:30:04.178702|
|2020-06-15 01:39:14.063628|
+--------------------------+
only showing top 5 rows


### Handle duplicates

In [7]:
df_users_bronze.count()


751501

In [8]:
total_rows     = df_users_bronze.count()
distinct_users = df_users_bronze.distinct().count()
duplicates     = total_rows - distinct_users

print(f"Total rows     : {total_rows:,}")
print(f"Distinct users : {distinct_users:,}")
print(f"Duplicates     : {duplicates:,}")

Total rows     : 751,501
Distinct users : 726,514
Duplicates     : 24,987


In [9]:
df_users_silver =df_users_bronze\
    .withColumn("user_first_touch_timestamp", (col("user_first_touch_timestamp").cast(LongType())/1000000)\
    .cast(TimestampType()))\
    .dropDuplicates()

In [10]:
#Verification
total_rows     = df_users_silver.count()
distinct_users = df_users_silver.distinct().count()
duplicates     = total_rows - distinct_users

print(f"Total rows     : {total_rows:,}")
print(f"Distinct users : {distinct_users:,}")
print(f"Duplicates     : {duplicates:,}")

Total rows     : 726,514
Distinct users : 726,514
Duplicates     : 0


In [12]:
display(df_users_silver)

,user_id,user_first_touch_timestamp,email
0,UA000000102357766,2020-06-15 04:49:02.754197,ynelson@hotmail.com
1,UA000000102358514,2020-06-15 06:04:21.540516,taylorhill@ochoa.biz
2,UA000000102358926,2020-06-15 06:29:09.747031,sarah9696@hotmail.com
3,UA000000102360897,2020-06-15 07:41:24.796845,moorescott@hotmail.com
4,UA000000102362558,2020-06-15 08:18:48.534848,taylorvincent@gmail.com
5,UA000000102362667,2020-06-15 08:21:15.165557,lindaclarke@becker-johnson.info
6,UA000000102364218,2020-06-15 08:46:59.880631,andrewstewart@hotmail.com
7,UA000000102364312,2020-06-15 08:48:37.409920,johnrocha@jones.org
8,UA000000102364623,2020-06-15 08:53:28.190632,ufrancis@gibbs-washington.com
9,UA000000102364694,2020-06-15 08:54:25.684639,dixonkaren@hotmail.com


### Writing to silver layer as a table

In [0]:
df_users_silver.write.mode("overwrite").format("delta").saveAsTable("ecom_clickstream.silver.users")

## Silver — Products
### Transformations Applied

**1. Type Casting — `price`**
The `price` column is stored as `String` we cast it to `DoubleType`

### Reading & displaying

In [0]:
df_products_bronze = parse_entity(bronze_all, "products")

df_products_bronze.printSchema()
display(df_products_bronze.limit(5))

### Duplicates

In [0]:
total_rows     = df_products_bronze.count()
distinct_users = df_products_bronze.select('item_id').distinct().count()
duplicates     = total_rows - distinct_users

print(f"Total rows     : {total_rows:,}")
print(f"Distinct users : {distinct_users:,}")
print(f"Duplicates     : {duplicates:,}")

No duplicates so let's put the right type for price

### Cast column price

In [0]:
df_products_silver = df_products_bronze\
    .withColumn('price',col("price").cast(DoubleType()))
df_products_silver.display()

### Writing to silver layer

In [0]:
df_products_silver.write.mode("overwrite").format("delta").saveAsTable("ecom_clickstream.silver.products")

##Silver — events

### Reading and displaying

In [0]:
df_events_bronze = parse_entity(bronze_all, "events").drop("_ingested_at", "_source_file")

df_events_bronze.printSchema()
display(df_events_bronze.limit(5))

### Duplicates

In [0]:
total_rows     = df_events_bronze.count()
distinct_events = df_events_bronze.distinct().count()
duplicates     = total_rows - distinct_events

print(f"Total rows     : {total_rows:,}")
print(f"Distinct users : {distinct_events:,}")
print(f"Duplicates     : {duplicates:,}")

In [0]:
df_events_silver = df_events_bronze \
    .withColumn("event_timestamp", (col("event_timestamp").cast(LongType()) / 1000000).cast(TimestampType())) \
    .withColumn("event_previous_timestamp", (col("event_previous_timestamp").cast(LongType()) / 1000000).cast(TimestampType())) \
    .withColumn("user_first_touch_timestamp", (col("user_first_touch_timestamp").cast(LongType()) / 1000000).cast(TimestampType()))\
    .dropDuplicates()

In [0]:
total_rows      = df_events_silver.count()
distinct_events = df_events_silver.select('user_id', 'event_timestamp').distinct().count()
duplicates      = total_rows - distinct_events

print(f"Total rows     : {total_rows:,}")
print(f"Distinct users : {distinct_events:,}")
print(f"Duplicates     : {duplicates:,}")

In [0]:
df_events_silver.display()

### Cast event timestamp

In [0]:
df_events_silver.display()

In [0]:
#OK, we can select the key and get the values like this.
df_events_bronze.select(col("geo.city")).show(3)

### Flattening Nested Structs — `geo` and `ecommerce`

Both `geo` and `ecommerce` are **Struct columns** — they contain nested fields
stored as a single column in the Bronze Layer.

In [0]:
df_events_silver = df_events_bronze \
    .withColumn("event_timestamp", (col("event_timestamp").cast(LongType()) / 1000000).cast(TimestampType())) \
    .withColumn("event_previous_timestamp", (col("event_previous_timestamp").cast(LongType()) / 1000000).cast(TimestampType())) \
    .withColumn("user_first_touch_timestamp", (col("user_first_touch_timestamp").cast(LongType()) / 1000000).cast(TimestampType()))\
    .select('*',col("ecommerce.*"))\
    .select('*',col("geo.*"))\
    .drop('geo','ecommerce')\
    .dropDuplicates()
df_events_silver.display()

### Exploding & Flattening Array Column — `items`

The `items` column is an **Array of Structs** — it contains a list of products
associated with each event. Unlike `geo` and `ecommerce` which are simple Structs,
`items` cannot be flattened with `.*` directly.

Since one event can contain **multiple items**, storing them in the same row as the event
would violate data modeling principles. We therefore extract them into a **separate Silver table**
`silver.events_items`


**Step 1 — `explode("items")`**
Each element of the Array becomes a separate row.

**Step 2 — `col("item.*")`**
Once exploded, each row contains a single item Struct.

### just a test to compare on the gold layer for a better understanding

### Why We Use `explode_outer()` Instead of `explode()`

When exploding the `items` array column, not all events contain purchased items.
Events like `page_view`, `search`, or `add_to_cart` have an **empty array** `[]` 
or `null` in the `items` column — only `purchase` events contain actual items.

Using `explode()` on this column causes a **silent data loss** :
```
user_id  | event_name  | items
UA000001 | purchase    | [item1, item2]  → 2 rows after explode ✅
UA000002 | page_view   | []              → 0 rows after explode ❌ lost !
UA000003 | add_to_cart | null            → 0 rows after explode ❌ lost !
```

This means all users who never purchased — the majority of users —
would be completely removed from the Silver events table.
This would make any funnel analysis impossible since we would only
see users who completed a purchase.

**The fix — `explode_outer()`**
`explode_outer()` preserves rows with empty arrays or null values,
returning `null` for the exploded column instead of dropping the row entirely :
```
user_id  | event_name  | item_id
UA000001 | purchase    | M_STAN_K    ✅ item preserved
UA000002 | page_view   | null        ✅ row preserved with null
UA000003 | add_to_cart | null        ✅ row preserved with null
```

> ⚠️ This is a common silent bug in data pipelines —
> `explode()` never raises an error, it simply drops rows silently.
> Always use `explode_outer()` when the array column can be empty or null.

In [0]:
df_events_silver = df_events_bronze \
    .withColumn("event_timestamp", (col("event_timestamp").cast(LongType()) / 1000000).cast(TimestampType())) \
    .withColumn("event_previous_timestamp", (col("event_previous_timestamp").cast(LongType()) / 1000000).cast(TimestampType())) \
    .withColumn("user_first_touch_timestamp", (col("user_first_touch_timestamp").cast(LongType()) / 1000000).cast(TimestampType()))\
    .select('*',col("ecommerce.*"))\
    .select('*',col("geo.*"))\
    .drop('geo','ecommerce')\
    .select('*',F.explode_outer("items").alias("ITEM"))\
    .select('*',col("ITEM.*")).drop('ITEM','items')\
    .dropDuplicates()
df_events_silver.display()

In [0]:
total_rows     = df_events_silver.count()
distinct_events = df_events_silver.distinct().count()
duplicates     = total_rows - distinct_events

print(f"Total rows     : {total_rows:,}")
print(f"Distinct users : {distinct_events:,}")
print(f"Duplicates     : {duplicates:,}")

In [0]:
df_events_silver.write.mode("overwrite").format("delta").saveAsTable("ecom_clickstream.silver.events")

### Writing to silver layer item & events

## Silver - Sales

### Reading & displaying

In [0]:
df_sales_bronze = parse_entity(bronze_all, "sales")

df_sales_bronze.printSchema()
display(df_sales_bronze.limit(5))

In [0]:
df_sales_bronze.select(
    F.count(F.when(col("transaction_timestamp").isNull(), 1)).alias("transaction_timestamp_nulls"),
    F.count(F.when(col("transactions_timestamp").isNull(), 1)).alias("transactions_timestamp_nulls")
).show()

In [0]:
df_sales_bronze.count()

### TIMESTAMP

In [0]:
df_sales_silver = df_sales_bronze\
    .withColumn("transaction_timestamp", F.coalesce(col("transaction_timestamp"),col("transactions_timestamp")))\
    .withColumn("transaction_timestamp", (col("transaction_timestamp").cast(LongType()) / 1000000).cast(TimestampType())) \
    .drop('transactions_timestamp')

display(df_sales_silver)

In [0]:
df_sales_silver.filter(col('transaction_timestamp').isNull()).count()
#No null on the column we created, seems good.

### Duplicates

In [0]:
df_sales_silver = df_sales_bronze\
    .withColumn("transaction_timestamp", F.coalesce(col("transaction_timestamp"),col("transactions_timestamp")))\
    .withColumn("transaction_timestamp", (col("transaction_timestamp").cast(LongType()) / 1000000).cast(TimestampType())) \
    .drop('transactions_timestamp')\
    .dropDuplicates()

In [0]:
total_rows     = df_sales_silver.count()
distinct_sales = df_sales_silver.distinct().count()
duplicates     = total_rows - distinct_sales

print(f"Total rows     : {total_rows:,}")
print(f"Distinct users : {distinct_sales:,}")
print(f"Duplicates     : {duplicates:,}")

### Exploding and Flattening — `items`

The `items` column contains a **nested structure** combining both an Array and Structs:
```
items = [                            ← Array (list of items)
  {                                  ← Struct (one item and its attributes)
    "coupon": null,
    "item_id": "M_STAN_Q",
    "item_name": "Standard Queen Mattress",
    "item_revenue_in_usd": 1045,
    "price_in_usd": 1045,
    "quantity": 1
  }
]
```
This requires **two operations** to fully flatten:

**Step 1 — `explode("items")`**
Transforms each element of the Array into a separate row.
A transaction with 3 items becomes 3 rows, each containing one item Struct.

**Step 2 — `col("item.*")`**
Flattens the Struct into individual columns

In the Bronze Layer, the `items` column was read as `StringType` because `inferSchema=false`
was applied during CSV ingestion. This prevents us from directly using `explode()` and `.*` 
flattening, as these operations require a `ArrayType`. To resolve this, I will cange the `inferSchema` directly to the bronze layer. 

PS : it doesnt change anything since the historical and users don't have the same `inferSchema`. Since it's not recommended to change anything in the bronze layer, I'll resolve it in the silver layer

PS 2 : When reading the `sales-csv` source, Spark could not automatically detect that the `items` column contains JSON — it is stored as plain text in the CSV file, so `inferSchema=true` defaults to `StringType`.In contrast, the `sales-historical` Parquet source already has a typed schema embedded,so Spark correctly reads `items` as `ArrayType` without any extra configuration.This is a fundamental limitation of the CSV format — it cannot natively store complex typeslike Arrays or Structs. Only JSON and Parquet support nested data types natively.

In [0]:
df_sales_silver.display()

[{"coupon":null,"item_id":"M_PREM_T","item_name":"Premium Twin Mattress","item_revenue_in_usd":2190,"price_in_usd":1095,"quantity":2}]

In [0]:
# ============================================
# STEP 1 — Define items schema
# ============================================
items_schema = ArrayType(StructType([
    StructField("coupon",               StringType(), True),
    StructField("item_id",              StringType(), True),
    StructField("item_name",            StringType(), True),
    StructField("item_revenue_in_usd",  DoubleType(), True),
    StructField("price_in_usd",         DoubleType(), True),
    StructField("quantity",             LongType(),   True)
]))


In [0]:
df_sales_silver = df_sales_bronze \
    .withColumn("transaction_timestamp", F.coalesce(col("transaction_timestamp"), col("transactions_timestamp"))) \
    .withColumn("transaction_timestamp", (col("transaction_timestamp").cast(LongType()) / 1000000).cast(TimestampType())) \
    .drop("transactions_timestamp") \
    .withColumn("items", F.from_json(col("items"), items_schema))\
    .select("*", explode("items").alias("item"))\
    .select('*',col("item.*"))\
    .drop('items','item')\
    .dropDuplicates(["order_id"])\
    .withColumn("purchase_revenue_in_usd", col("purchase_revenue_in_usd").cast(DoubleType())) \
    .withColumn("total_item_quantity", col("total_item_quantity").cast(LongType())) \
    .withColumn("unique_items", col("unique_items").cast(LongType())) \

df_sales_silver.display()

### Writing to silver layer

In [0]:
df_sales_silver.write.mode("overwrite").format("delta").saveAsTable("ecom_clickstream.silver.sales")

PS : I had to get rid of ingested at and source file since I have duplicates I couldnt get deleted

PS2 : on the sales and sales historical the timestamp columns doesnt have the same name... (transactions_timestamp and transaction_timestamp). So I had the value for transactions on one side but not on the historical parts, even thought we have the information on the historical files. So the problem come from the moment we append the historical with the current data.

### Handling Inconsistent Column Names — `transaction_timestamp` vs `transactions_timestamp`

During the Bronze ingestion, two sources were appended into the same `sales` table:
- `sales-csv` contains a column named `transactions_timestamp` (with an **'s'**)
- `sales-historical` contains a column named `transaction_timestamp` (without **'s'**)

Since Delta Lake's `mergeSchema` never automatically maps columns with different names,
it created **two separate columns**, filling with `null` where the column did not exist in the source:
```
order_id | transactions_timestamp | transaction_timestamp
---------|------------------------|----------------------
306148   | 1592677646393617       | null                   ← from CSV
306148   | null                   | 1592677646393617       ← from historical
```

This is a classic example of **naming inconsistency between sources** — very common in
real-world data engineering projects where different teams produce data independently.

The Bronze Layer captures this faithfully as-is, and the Silver Layer reconciles it
using `coalesce()`, which returns the first non-null value between the two columns:
```python
coalesce(col("transaction_timestamp"), col("transactions_timestamp"))
```

This produces a single clean `transaction_timestamp` column with no nulls,
and the redundant `transactions_timestamp` column is then dropped.